In [ ]:
import torch
from torch import nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(d_model, d_model)

    def _rotate_half(self, x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def _rope(self, q, k, position_ids=None, base=10000):
        if position_ids is None:
            seq_len = q.size(-2)
            position_ids = torch.arange(seq_len, device=q.device)[None, :]

        freqs = torch.arange(0, self.d_k, 2, device=q.device) / self.d_k
        freqs = base ** -freqs
        angles = position_ids[..., None] * freqs[None, None, :]
        sin = angles.sin().repeat_interleave(2, dim=-1)[:, None, :, :]
        cos = angles.cos().repeat_interleave(2, dim=-1)[:, None, :, :]
        return q * cos + self._rotate_half(q) * sin, \
              k * cos + self._rotate_half(k) * sin

    def forward(self, x, position_ids=None, seq_idx=None):
        batch_size, seq_len, _ = x.shape

        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        Q, K = self._rope(Q, K, position_ids=position_ids)
        score = Q @ K.transpose(-2, -1) / self.d_k ** 0.5

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1
        )
        mask_value = torch.finfo(score.dtype).min
        score = score.masked_fill(causal_mask[None, None, :, :], mask_value)

        if seq_idx is not None:
            same_seq_mask = seq_idx[:, :, None] == seq_idx[:, None, :]
            score = score.masked_fill(~same_seq_mask[:, None, :, :], mask_value)

        attn_w = torch.softmax(score, dim=-1)
        output = self.dropout(attn_w) @ V
        output = output.transpose(1, 2).reshape(batch_size, -1, self.d_model)
        output = self.out(output)
        return output


class TransformerDecoderBlock(nn.Module):
    def __init__(self, ffn_dim, d_model, num_heads, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, position_ids=None, seq_idx=None):
        x_norm = self.norm1(x)
        x = x + self.dropout(self.attn(x_norm, position_ids=position_ids, seq_idx=seq_idx))
        x_norm = self.norm2(x)
        x = x + self.dropout(self.ffn(x_norm))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, output_dim, ffn_dim, d_model, num_heads, num_layers, dropout=0.1, padding_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(ffn_dim, d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.out = nn.Linear(d_model, output_dim)

    def forward(self, x, position_ids=None, seq_idx=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x, position_ids=position_ids, seq_idx=seq_idx)
        return self.out(x)

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithFlattening

batch_size = 16
subset_size = 50000
max_length = 256
dataset = load_dataset("roneneldan/TinyStories")

tokenizer = AutoTokenizer.from_pretrained("gpt2")

subset = dataset["train"].select(range(subset_size))

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=max_length)

tokenized_dataset = subset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing"
)

collate_fn = DataCollatorWithFlattening(
    return_position_ids=True,
    return_seq_idx=True,
    return_tensors="pt"
)

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True,
    num_workers=4
)

len(train_dataloader)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 1
vocab_size = tokenizer.vocab_size
model = TransformerDecoder(
    vocab_size=vocab_size,
    output_dim=vocab_size,
    ffn_dim=256,
    d_model=128,
    num_heads=8,
    num_layers=6,
    dropout=0.1,
    padding_idx=tokenizer.pad_token_id,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, fused=True)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(device)

for i in range(epochs):
    for idx, data in enumerate(train_dataloader):
        input_ids = data["input_ids"].to(device)
        labels = data["labels"].to(device)
        position_ids = data["position_ids"].to(device)
        seq_idx = data["seq_idx"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device.type):
            logits = model(input_ids, position_ids=position_ids, seq_idx=seq_idx)
            loss = criterion(
                logits[:, :-1, :].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
            )
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if (idx+1) % 10 == 0:
            print(f"Epoch {i+1}, Step {idx+1}, Loss: {loss.item():.4f}, PPL: {torch.exp(loss).item():.4f}")

In [ ]:

@torch.no_grad()
def generate(model, tokenizer, prompt, max_length, temperature=1.0, top_k=50, device="cuda"):
    model.eval()
    
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    
    for _ in range(max_length):
        logits = model(input_ids)
        next_logits = logits[:, -1, :] / temperature
        
        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            next_logits = torch.full_like(next_logits, float('-inf'))
            next_logits.scatter_(1, indices, values)
        
        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        input_ids = torch.cat([input_ids, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return generated_text

model.eval()
prompt = "A long time ago"
story = generate(model, tokenizer, prompt, max_length, temperature=0.8, top_k=50, device=device)
print(story)
